In [1]:
with open ('MyFitbitData/FadiE/Sleep/Computed Temperature - 2022-04-01.csv') as f:
    lines = f.readlines()
    for line in lines:
        print(line.strip())

type,sleep_start,sleep_end,temperature_samples,nightly_temperature,baseline_relative_sample_sum,baseline_relative_sample_sum_of_squares,baseline_relative_nightly_standard_deviation,baseline_relative_sample_standard_deviation
IDT,2022-04-05T23:12:30,2022-04-06T04:44:30,332,32.024096385542165,NaN,NaN,NaN,NaN
IDT,2022-04-06T23:28,2022-04-07T04:40:30,313,31.723961661341853,NaN,NaN,NaN,NaN
IDT,2022-04-07T23:24,2022-04-08T03:37,253,32.548221343873514,132.60361445783292,456.35256350704003,0.41717139740236076,1.2390019338493006
IDT,2022-04-08T23:52,2022-04-09T04:29,277,32.433574007220216,56.71265060241058,675.6490423767827,0.37953452464209697,1.4184696760191007
IDT,2022-04-09T23:21,2022-04-10T04:19,298,32.636577181208054,60.49494584837703,645.8519747422542,0.386366038850727,1.434791588735201
IDT,2022-04-11T00:06,2022-04-11T04:10:30,245,32.90897959183674,102.43006949101891,505.82440463645617,0.432173582433223,1.4221217511685422
IDT,2022-04-11T23:18,2022-04-12T03:48:30,271,32.618450184501846,19.

In [2]:
import pandas as pd
import plotly.express as px

df = pd.read_csv('MyFitbitData/FadiE/Sleep/Computed Temperature - 2022-04-01.csv')
fig = px.line(df, x='sleep_start', y='nightly_temperature', title='Temperature over Time')
fig.show()


In [ ]:
import os
import pandas as pd

import plotly.express as px

# Directory containing the CSV files
directory = 'MyFitbitData/FadiE/Sleep/'

# List to hold dataframes
dfs = []

# Loop through all files in the directory
for filename in os.listdir(directory):
    if filename.startswith("Computed Temperature") and filename.endswith(".csv"):
        filepath = os.path.join(directory, filename)
        df_temp = pd.read_csv(filepath)
        dfs.append(df_temp)

# Concatenate all dataframes
df_large = pd.concat(dfs, ignore_index=True)

# Plot the resulting dataframe
fig_large = px.line(df_large[['sleep_start', 'nightly_temperature']].groupby('sleep_start').mean().reset_index(), x='sleep_start', y='nightly_temperature', title='Temperature over Time')
fig_large.show()

In [ ]:
# Convert sleep_start and sleep_end to datetime
df_large['sleep_start'] = pd.to_datetime(df_large['sleep_start'], errors='coerce')
df_large['sleep_end'] = pd.to_datetime(df_large['sleep_end'], errors='coerce')

# Calculate sleep_duration in hours
df_large['sleep_duration'] = (df_large['sleep_end'] - df_large['sleep_start']).dt.total_seconds() / 3600



            sleep_start           sleep_end  sleep_duration
0   2023-03-31 23:40:30 2023-04-01 04:47:30        5.116667
1                   NaT 2023-04-02 04:30:30             NaN
2   2023-04-02 23:50:30                 NaT             NaN
3                   NaT 2023-04-04 04:39:30             NaN
4                   NaT                 NaT             NaN
..                  ...                 ...             ...
737                 NaT 2023-01-28 05:19:30             NaN
738 2023-01-29 00:26:30                 NaT             NaN
739 2023-01-30 00:33:30                 NaT             NaN
740 2023-01-31 00:36:30 2023-01-31 06:18:30        5.700000
741                 NaT 2023-02-01 02:53:30             NaN

[742 rows x 3 columns]


In [13]:
# Drop rows where sleep_start or sleep_end is null
df_large = df_large.dropna(subset=['sleep_start', 'sleep_end'])

# Sort the dataframe by sleep_start
df_large = df_large.sort_values(by='sleep_start')

# Recalculate sleep_duration in hours
df_large['sleep_duration'] = (df_large['sleep_end'] - df_large['sleep_start']).dt.total_seconds() / 3600

# Calculate the rolling sum of sleep_duration over 7 days
df_large['rolling_sleep_duration'] = df_large['sleep_duration'].rolling(window=7).mean()

# Display the updated dataframe with rolling average
fig_large = px.line(df_large[['sleep_start', 'rolling_sleep_duration']].dropna(), x='sleep_start', y='rolling_sleep_duration', title='7-Day Rolling Sum of Sleep Duration over Time')
fig_large.show()